# File Process

1. Check files for headers and print to verify 
2. Union, filter Jan 2012 - Dec 2016
3. Keep all columns

In [1]:
import csv
import glob
import os
import duckdb


DATA_DIR = os.path.join("data", "Raw")
COMBINED_DIR = os.path.join("data", "Combined")
os.makedirs(COMBINED_DIR, exist_ok=True)

# Date Range filter
START_MONTH = "2012-01"
END_MONTH = "2016-12"
OUTPUT_FILE = os.path.join(COMBINED_DIR, "resale_flat_prices_2012_01_to_2016_12.parquet")

## Get all files in the folder and filer

In [2]:
# Get all files in the folder
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

# Filter data file using date
files_to_process = []
for file in all_files:
    if file != OUTPUT_FILE:
        files_to_process.append(file)
        
files_to_process.sort()

print("Column header of files :")
all_headers = []

# Prints headers only
for file_path in files_to_process:
    
    
    with open(file_path, 'r') as f:
        first_line = f.readline()
        
    
        header = first_line.strip().split(',')
        
        # print header
        file_name = os.path.basename(file_path)
        print(f"  {file_name}: {header}")
        
        all_headers.append(header)

Column header of files :
  ResaleFlatPricesBasedonApprovalDate19901999.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']
  ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_s

## superset of columns in all files 

In [3]:
union_columns = []
for header in all_headers:
    for col in header:
        if col not in union_columns:
            union_columns.append(col)

## filter data and write to Parquet 

In [4]:
# Replace with forward slashes to avoid regex issues 
files_input = [f.replace('\\', '/') for f in files_to_process]
print(f" Input files are: {files_input}")
output_parquet = OUTPUT_FILE.replace('\\', '/')
cols_str = ', '.join(union_columns)

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

# filter data 
duckdb.sql(f"""
    COPY (
        SELECT {cols_str}
        FROM read_csv_auto({files_input}, union_by_name=True)
        WHERE month >= '{START_MONTH}' AND month <= '{END_MONTH}'
        ORDER BY month
    ) TO '{output_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# row count and file size
file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"Passed. Wrote to {OUTPUT_FILE} ({file_size_mb:.2f} MB)\n")


 Input files are: ['data/Raw/ResaleFlatPricesBasedonApprovalDate19901999.csv', 'data/Raw/ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv', 'data/Raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv', 'data/Raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv', 'data/Raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv']


Passed. Wrote to data\Combined\resale_flat_prices_2012_01_to_2016_12.parquet (0.59 MB)

